# Demo 6: Research-Backed AI-Music Improvements

This notebook keeps the parent paper's MFCC-plus-CNN pipeline and the Demo 5 AI-inference comparison, then adds a research-backed classical machine-learning block. The new contribution uses MFCC statistical summaries and lightweight classifiers, with a linear SVM selected because it improved AI-music track accuracy in the local comparison while staying easy to explain in the final paper.

## Notebook Roadmap
1. configure local paths and experiment settings
2. inspect one GTZAN example through waveform, rectification, smoothing, FFT, half-spectrum, STFT, spectrogram, and MFCC views
3. build MFCC segment datasets from the local GTZAN folders
4. import Keras separately and define the paper-style CNN
5. train and evaluate the model on GTZAN
6. optionally evaluate the same model on local Suno AI music

## Part 1: Setup And Local Paths

This section keeps everything local. GTZAN is read from `data/genres_original`, Suno AI music is read from `data/suno-audio`, and all outputs are written under `artifacts/demo_6`.

The main classification settings stay aligned with the paper replication, while the novel AI-music and research-backed classifier comparisons write their own result tables under `artifacts/demo_6`: `22050 Hz` sample rate, `13` MFCC coefficients, `2048` FFT window, `512` hop length, and `5` segments per track.

In [ ]:
import json
import math
import os
import tempfile
import time
import warnings
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "demo_6"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "demo_3"

MPLCONFIG_DIR = ARTIFACT_DIR / ".mplconfig"
MPLCONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIG_DIR))
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

warnings.filterwarnings("ignore", message="PySoundFile failed. Trying audioread instead.")
warnings.filterwarnings("ignore", category=FutureWarning, message=".*audioread_load.*")

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Audio as JupyterAudio, display
from datasets import Audio, load_from_disk
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

GTZAN_RAW_DIR = PROJECT_ROOT / "data" / "genres_original"
AI_DATA_ROOT = PROJECT_ROOT / "data" / "suno-audio"

GTZAN_DATA_JSON = ARTIFACT_DIR / "gtzan_data.json"
MODEL_PATH = ARTIFACT_DIR / "paper_cnn.keras"
HISTORY_CSV = ARTIFACT_DIR / "gtzan_training_history.csv"
GTZAN_METRICS_JSON = ARTIFACT_DIR / "gtzan_metrics.json"
AI_JSON_PATH = ARTIFACT_DIR / "suno_ai_eval_mfcc.json"
AI_METADATA_CSV = ARTIFACT_DIR / "suno_ai_eval_metadata.csv"
AI_SEGMENT_RESULTS_CSV = ARTIFACT_DIR / "ai_segment_results.csv"
AI_TRACK_RESULTS_CSV = ARTIFACT_DIR / "ai_track_results.csv"
AI_METHOD_COMPARISON_CSV = ARTIFACT_DIR / "ai_method_comparison.csv"
AI_TRACK_METHOD_RESULTS_CSV = ARTIFACT_DIR / "ai_track_method_results.csv"
AI_TTA_JSON_PATH = ARTIFACT_DIR / "suno_ai_tta_mfcc.json"
AI_TTA_VIEW_RESULTS_CSV = ARTIFACT_DIR / "ai_tta_view_results.csv"
RESEARCH_METHOD_COMPARISON_CSV = ARTIFACT_DIR / "research_method_comparison.csv"
RESEARCH_TRACK_RESULTS_CSV = ARTIFACT_DIR / "research_track_results.csv"
RESEARCH_SEGMENT_RESULTS_CSV = ARTIFACT_DIR / "research_segment_results.csv"
SOURCE_GTZAN_DATA_JSON = SOURCE_ARTIFACT_DIR / "gtzan_data.json"
SOURCE_MODEL_PATH = SOURCE_ARTIFACT_DIR / "paper_cnn.keras"
SOURCE_HISTORY_CSV = SOURCE_ARTIFACT_DIR / "gtzan_training_history.csv"
SOURCE_AI_JSON_PATH = SOURCE_ARTIFACT_DIR / "suno_ai_eval_mfcc.json"

SAMPLE_RATE = 22050
TRACK_DURATION = 30
SAMPLES_PER_TRACK = SAMPLE_RATE * TRACK_DURATION
NUM_MFCC = 13
N_FFT = 2048
HOP_LENGTH = 512
NUM_SEGMENTS = 5
SMOOTHING_WINDOW = 3000

TEST_SIZE = 0.3
VALIDATION_SIZE = 0.2
LEARNING_RATE = 1e-4
EPOCHS = 30
BATCH_SIZE = 32
SEED = 42

REBUILD_GTZAN_JSON = not GTZAN_DATA_JSON.exists() and not SOURCE_GTZAN_DATA_JSON.exists()
RETRAIN_MODEL = False
STRICT_SINGLE_LABEL = True
MAX_AI_TRACKS_PER_GENRE = 30
REBUILD_AI_FEATURES = False
REBUILD_AI_TTA_FEATURES = False
RUN_AI_EVAL = AI_DATA_ROOT.exists()
RUN_AI_TTA_EVAL = False
TTA_NUM_VIEWS = 9
TTA_WINDOW_SECONDS = TRACK_DURATION / NUM_SEGMENTS

EXAMPLE_GENRE = "blues"
EXAMPLE_GENRE_DIR = GTZAN_RAW_DIR / EXAMPLE_GENRE

if not GTZAN_RAW_DIR.exists():
    raise FileNotFoundError("Could not find the local GTZAN folder at data/genres_original.")

example_files = sorted(EXAMPLE_GENRE_DIR.glob("*.wav"))
if not example_files:
    raise FileNotFoundError(f"Could not find example audio files for the '{EXAMPLE_GENRE}' genre.")
EXAMPLE_FILE = example_files[0]

print(f"GTZAN folder exists: {GTZAN_RAW_DIR.exists()} (expected: data/genres_original)")
print(f"AI data root exists: {AI_DATA_ROOT.exists()} (expected: data/suno-audio)")
print(f"Example genre: {EXAMPLE_GENRE}")
print(f"Example file: {EXAMPLE_FILE.name}")
print("Artifacts folder: artifacts/demo_6")
print(f"Fallback baseline artifacts exist: {SOURCE_ARTIFACT_DIR.exists()} (expected: artifacts/demo_3)")
print(f"Will rebuild GTZAN MFCC JSON: {REBUILD_GTZAN_JSON}")
print(f"Will run AI evaluation: {RUN_AI_EVAL}")

## Part 2: Load One Example Track

The paper explains its preprocessing steps by taking one song and transforming it step by step. This cell loads one local GTZAN track twice: once at its original sample rate for reference, and once resampled to `22050 Hz` so it matches the later MFCC pipeline.

In [ ]:
raw_signal, raw_sr = librosa.load(EXAMPLE_FILE, sr=None)
signal, sr = librosa.load(EXAMPLE_FILE, sr=SAMPLE_RATE)

track_summary = pd.DataFrame(
    {
        "value": [
            EXAMPLE_GENRE,
            EXAMPLE_FILE.name,
            raw_sr,
            sr,
            round(len(raw_signal) / raw_sr, 2),
            round(len(signal) / sr, 2),
            signal.shape,
        ]
    },
    index=[
        "genre",
        "file",
        "original_sample_rate",
        "analysis_sample_rate",
        "original_duration_seconds",
        "analysis_duration_seconds",
        "analysis_signal_shape",
    ],
)

display(track_summary)
display(JupyterAudio(signal, rate=sr))

## Part 3: Original Waveform And Rectified Signal

Rectification replaces each sample with its absolute value. That makes the amplitude envelope easier to inspect because negative and positive oscillations no longer cancel each other out visually.

In [ ]:
rectified_signal = np.abs(signal)
time_axis = np.arange(len(signal)) / sr

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
librosa.display.waveshow(signal, sr=sr, ax=axes[0], color="steelblue")
axes[0].set_title("Original Waveform")
axes[0].set_ylabel("Amplitude")

axes[1].plot(time_axis, rectified_signal, color="darkorange", linewidth=0.8)
axes[1].set_title("Rectified Signal")
axes[1].set_xlabel("Time (seconds)")
axes[1].set_ylabel("Absolute amplitude")
plt.tight_layout()
plt.show()

waveform_stats = pd.DataFrame(
    {
        "value": [
            float(signal.min()),
            float(signal.max()),
            float(signal.mean()),
            float(rectified_signal.mean()),
        ]
    },
    index=["min_amplitude", "max_amplitude", "mean_amplitude", "mean_absolute_amplitude"],
)

display(waveform_stats)

## Part 4: Smooth The Rectified Signal

A rolling average turns the sharply oscillating rectified waveform into a smoother amplitude envelope. This is mainly useful for exploratory understanding rather than as a direct model input in the paper's final classifier.

In [ ]:
smoothed_signal = (
    pd.Series(rectified_signal)
    .rolling(window=SMOOTHING_WINDOW, min_periods=1)
    .mean()
    .to_numpy()
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(time_axis, rectified_signal, color="lightgray", linewidth=0.7, label="rectified")
ax.plot(time_axis, smoothed_signal, color="crimson", linewidth=1.5, label="smoothed envelope")
ax.set_title("Rectified Signal With Rolling-Window Smoothing")
ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Amplitude")
ax.legend()
plt.tight_layout()
plt.show()

## Part 5: FFT And Half Power Spectrum

The Fast Fourier Transform moves the signal from the time domain into the frequency domain. The half-spectrum keeps only non-negative frequencies, which is the more interpretable view for real-valued audio signals.

In [ ]:
fft = np.fft.fft(signal)
frequencies = np.fft.fftfreq(len(signal), d=1 / sr)
power_spectrum = np.abs(fft) ** 2

positive_mask = frequencies >= 0
half_frequencies = frequencies[positive_mask]
half_power_spectrum = power_spectrum[positive_mask]

dominant_frequency_hz = float(half_frequencies[np.argmax(half_power_spectrum[1:]) + 1])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(frequencies, power_spectrum, color="navy", linewidth=0.7)
axes[0].set_title("Full Power Spectrum")
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("Power")
axes[0].set_xlim(frequencies.min(), frequencies.max())

axes[1].plot(half_frequencies, half_power_spectrum, color="seagreen", linewidth=0.8)
axes[1].set_title("Half Power Spectrum")
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Power")
axes[1].set_xlim(0, 8000)
plt.tight_layout()
plt.show()

fft_summary = pd.DataFrame(
    {"value": [dominant_frequency_hz, float(half_power_spectrum.max())]},
    index=["dominant_frequency_hz", "max_half_spectrum_power"],
)
display(fft_summary)

## Part 6: STFT, Spectrogram, And dB Spectrogram

The Short-Time Fourier Transform computes a local FFT over many overlapping windows. That lets us track how the frequency content changes over time, which is why it becomes a spectrogram instead of a single global frequency profile.

In [ ]:
stft = librosa.stft(signal, n_fft=N_FFT, hop_length=HOP_LENGTH)
stft_magnitude = np.abs(stft)
stft_db = librosa.amplitude_to_db(stft_magnitude, ref=np.max)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

img1 = librosa.display.specshow(
    stft_magnitude,
    sr=sr,
    hop_length=HOP_LENGTH,
    x_axis="time",
    y_axis="log",
    ax=axes[0],
)
axes[0].set_title("STFT Magnitude Spectrogram")
fig.colorbar(img1, ax=axes[0], format="%+2.0f")

img2 = librosa.display.specshow(
    stft_db,
    sr=sr,
    hop_length=HOP_LENGTH,
    x_axis="time",
    y_axis="log",
    cmap="magma",
    ax=axes[1],
)
axes[1].set_title("Spectrogram In dB Scale")
fig.colorbar(img2, ax=axes[1], format="%+2.0f dB")

plt.tight_layout()
plt.show()

## Part 7: MFCCs And The Bridge To The Classifier

MFCCs are the compact features that the parent paper actually feeds into the CNN. The earlier waveform, smoothing, FFT, and spectrogram steps help explain the signal, but the classification pipeline still stays faithful to the paper by using MFCC segments as the final model input.

In [ ]:
mfcc = librosa.feature.mfcc(
    y=signal,
    sr=sr,
    n_mfcc=NUM_MFCC,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
)

fig, ax = plt.subplots(figsize=(14, 5))
img = librosa.display.specshow(mfcc, x_axis="time", sr=sr, hop_length=HOP_LENGTH, ax=ax)
ax.set_title("MFCC Representation")
fig.colorbar(img, ax=ax)
plt.tight_layout()
plt.show()

preprocessing_summary = pd.DataFrame(
    {
        "value": [
            signal.shape,
            rectified_signal.shape,
            smoothed_signal.shape,
            stft.shape,
            stft_magnitude.shape,
            stft_db.shape,
            mfcc.shape,
            NUM_MFCC,
            N_FFT,
            HOP_LENGTH,
        ]
    },
    index=[
        "signal_shape",
        "rectified_shape",
        "smoothed_shape",
        "stft_complex_shape",
        "stft_magnitude_shape",
        "stft_db_shape",
        "mfcc_shape",
        "num_mfcc",
        "n_fft",
        "hop_length",
    ],
)

display(preprocessing_summary)

## Part 8: Build The GTZAN MFCC Dataset

This is where the exploratory example turns into a dataset pipeline. Each GTZAN song is split into equal segments, MFCCs are extracted from each segment, and the results are stored in a JSON file so later notebook runs can skip the expensive preprocessing step.

In [ ]:
def extract_mfcc_segments_from_file(file_path, num_mfcc=NUM_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH, num_segments=NUM_SEGMENTS):
    signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    samples_per_segment = int(SAMPLES_PER_TRACK / num_segments)
    expected_vectors = math.ceil(samples_per_segment / hop_length)

    segments = []
    for segment_idx in range(num_segments):
        start_sample = samples_per_segment * segment_idx
        finish_sample = start_sample + samples_per_segment
        segment_signal = signal[start_sample:finish_sample]

        if len(segment_signal) < n_fft:
            continue

        mfcc = librosa.feature.mfcc(
            y=segment_signal,
            sr=sr,
            n_mfcc=num_mfcc,
            n_fft=n_fft,
            hop_length=hop_length,
        ).T

        if len(mfcc) == expected_vectors:
            segments.append(mfcc)

    return segments


def save_mfcc_from_folder(dataset_path, json_path, num_mfcc=NUM_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH, num_segments=NUM_SEGMENTS):
    dataset_path = Path(dataset_path)
    data = {"mapping": [], "labels": [], "mfcc": []}

    for genre_idx, genre_dir in enumerate(sorted(p for p in dataset_path.iterdir() if p.is_dir())):
        data["mapping"].append(genre_dir.name)
        print(f"Processing GTZAN genre: {genre_dir.name}")

        for audio_file in sorted(genre_dir.iterdir()):
            if audio_file.suffix.lower() not in {".wav", ".mp3", ".au", ".flac", ".ogg", ".m4a"}:
                continue

            try:
                segments = extract_mfcc_segments_from_file(audio_file, num_mfcc, n_fft, hop_length, num_segments)
            except Exception as exc:
                relative_audio_path = audio_file.relative_to(dataset_path)
                print(f"SKIP (cannot read): {relative_audio_path} | {type(exc).__name__}")
                continue

            for mfcc in segments:
                data["mfcc"].append(mfcc.tolist())
                data["labels"].append(genre_idx)

    with open(json_path, "w") as fp:
        json.dump(data, fp)

    return data


def load_mfcc_json(json_path):
    with open(json_path, "r") as fp:
        data = json.load(fp)
    X = np.array(data["mfcc"], dtype=np.float32)
    y = np.array(data["labels"], dtype=np.int64)
    return data, X, y


def prepare_datasets_from_json(json_path, test_size=TEST_SIZE, validation_size=VALIDATION_SIZE, random_state=SEED):
    data, X, y = load_mfcc_json(json_path)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=validation_size, random_state=random_state)

    X_train = X_train[..., np.newaxis]
    X_val = X_val[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    return data["mapping"], X_train, X_val, X_test, y_train, y_val, y_test


def plot_confusion_matrix(y_true, y_pred, label_names, title):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(label_names))))
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(label_names)))
    ax.set_yticks(range(len(label_names)))
    ax.set_xticklabels(label_names, rotation=45, ha="right")
    ax.set_yticklabels(label_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    for i in range(len(label_names)):
        for j in range(len(label_names)):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
    fig.colorbar(im)
    plt.tight_layout()
    plt.show()

## Part 9: Import Keras And Define The Paper-Style CNN

Keras is imported in its own cell because it is usually the heaviest import in the notebook. The model definition below matches the same Conv2D, MaxPooling, BatchNormalization, Dense, and Dropout pattern used in the paper replication.

In [ ]:
t = time.time()
import keras
keras.utils.set_random_seed(SEED)
print(f"Keras version: {keras.__version__}")
print(f"Keras import finished in {time.time() - t:.2f} seconds")


def build_paper_cnn(input_shape, num_classes):
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=input_shape))
    model.add(keras.layers.Conv2D(32, (3, 3), activation="relu"))
    model.add(keras.layers.MaxPooling2D((3, 3), strides=(2, 2), padding="same"))
    model.add(keras.layers.BatchNormalization())
    model.add(keras.layers.Conv2D(32, (3, 3), activation="relu"))
    model.add(keras.layers.MaxPooling2D((3, 3), strides=(2, 2), padding="same"))
    model.add(keras.layers.BatchNormalization())
    model.add(keras.layers.Conv2D(32, (2, 2), activation="relu"))
    model.add(keras.layers.MaxPooling2D((2, 2), strides=(2, 2), padding="same"))
    model.add(keras.layers.BatchNormalization())
    model.add(keras.layers.Flatten())
    model.add(keras.layers.Dense(64, activation="relu"))
    model.add(keras.layers.Dropout(0.3))
    model.add(keras.layers.Dense(num_classes, activation="softmax"))
    return model

## Part 10: Train And Evaluate The Model On Local GTZAN

This section performs the actual model implementation. If the MFCC JSON already exists, it reuses it. Otherwise it rebuilds the features from your local GTZAN folders. The trained model is saved so later runs can skip retraining unless `RETRAIN_MODEL = True`.

In [ ]:
if REBUILD_GTZAN_JSON:
    print("Building GTZAN MFCC JSON from local audio...")
    save_mfcc_from_folder(GTZAN_RAW_DIR, GTZAN_DATA_JSON)

gtzan_json_input_path = GTZAN_DATA_JSON if GTZAN_DATA_JSON.exists() else SOURCE_GTZAN_DATA_JSON
if not gtzan_json_input_path.exists():
    raise FileNotFoundError("Could not find or build a GTZAN MFCC JSON file for this notebook.")

label_names, X_train, X_val, X_test, y_train, y_val, y_test = prepare_datasets_from_json(gtzan_json_input_path)
genre_to_id = {genre: idx for idx, genre in enumerate(label_names)}

print("Using GTZAN MFCC JSON:", gtzan_json_input_path.relative_to(PROJECT_ROOT))
print("Label order:", label_names)
print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)

model_input_path = MODEL_PATH if MODEL_PATH.exists() else SOURCE_MODEL_PATH
history_input_path = HISTORY_CSV if HISTORY_CSV.exists() else SOURCE_HISTORY_CSV

if model_input_path.exists() and not RETRAIN_MODEL:
    print("Loading baseline model:", model_input_path.relative_to(PROJECT_ROOT))
    model = keras.models.load_model(model_input_path)
    history_df = pd.read_csv(history_input_path) if history_input_path.exists() else pd.DataFrame()
else:
    model = build_paper_cnn(X_train.shape[1:], len(label_names))
    optimiser = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    model.compile(optimizer=optimiser, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1,
    )
    model.save(MODEL_PATH)
    history_df = pd.DataFrame(history.history)
    history_df.to_csv(HISTORY_CSV, index=False)

model.summary()
display(history_df.tail())

if not history_df.empty and {"accuracy", "val_accuracy", "loss", "val_loss"}.issubset(history_df.columns):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(history_df.index + 1, history_df["accuracy"], label="train")
    axes[0].plot(history_df.index + 1, history_df["val_accuracy"], label="validation")
    axes[0].set_title("Accuracy By Epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(history_df.index + 1, history_df["loss"], label="train")
    axes[1].plot(history_df.index + 1, history_df["val_loss"], label="validation")
    axes[1].set_title("Loss By Epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

gtzan_test_loss, gtzan_test_acc = model.evaluate(X_test, y_test, verbose=0)
gtzan_test_pred = model.predict(X_test, verbose=0).argmax(axis=1)
gtzan_report = classification_report(
    y_test,
    gtzan_test_pred,
    labels=list(range(len(label_names))),
    target_names=label_names,
    zero_division=0,
)

metrics_payload = {
    "gtzan_data_source": str(gtzan_json_input_path.relative_to(PROJECT_ROOT)),
    "model_source": str(model_input_path.relative_to(PROJECT_ROOT)) if model_input_path.exists() else str(MODEL_PATH.relative_to(PROJECT_ROOT)),
    "gtzan_test_loss": float(gtzan_test_loss),
    "gtzan_test_accuracy": float(gtzan_test_acc),
    "gtzan_test_macro_f1": float(f1_score(y_test, gtzan_test_pred, average="macro")),
}
with open(GTZAN_METRICS_JSON, "w") as fp:
    json.dump(metrics_payload, fp, indent=2)

print(f"GTZAN test accuracy: {gtzan_test_acc:.4f}")
print(f"GTZAN test macro F1: {metrics_payload['gtzan_test_macro_f1']:.4f}")
print(gtzan_report)
plot_confusion_matrix(y_test, gtzan_test_pred, label_names, "GTZAN Test Confusion Matrix")


## Part 11: Optional AI-Generated Music Evaluation

This final section reuses the trained GTZAN model on the local Suno AI dataset. Because Suno tracks have free-form tags instead of GTZAN labels, the notebook uses a simple tag-to-genre mapping. Keep in mind that this is a project extension rather than part of the original paper.

In [ ]:
GENRE_ALIASES = {
    "blues": {"blues"},
    "classical": {"classical"},
    "country": {"country"},
    "disco": {"disco"},
    "hiphop": {"hip hop", "hiphop"},
    "jazz": {"jazz"},
    "metal": {"metal"},
    "pop": {"pop"},
    "reggae": {"reggae"},
    "rock": {"rock"},
}


def infer_track_keys_from_segment_order(track_ids, max_views_per_track=NUM_SEGMENTS):
    """Recover per-row track keys when older cached AI JSON only has track IDs."""
    keys = []
    current_track_id = None
    current_view_count = 0
    current_chunk = 0
    for raw_track_id in track_ids:
        track_id = str(raw_track_id)
        if track_id != current_track_id:
            current_track_id = track_id
            current_view_count = 0
            current_chunk = 0
        elif current_view_count >= max_views_per_track:
            current_view_count = 0
            current_chunk += 1
        keys.append(f"{track_id}__chunk_{current_chunk}")
        current_view_count += 1
    return keys


def normalize_tags(raw_tags):
    return {
        " ".join(part.strip().lower().split())
        for part in str(raw_tags).split(",")
        if part and part.strip()
    }


def map_tags_to_gtzan_genre(raw_tags, strict_single_label=True):
    tags = normalize_tags(raw_tags)
    matched = []
    for genre, aliases in GENRE_ALIASES.items():
        if any(alias in tags for alias in aliases):
            matched.append(genre)
    matched = sorted(set(matched))
    if len(matched) == 1:
        return matched[0]
    if strict_single_label:
        return None
    return matched[0] if matched else None


def collect_ai_eval_metadata(ai_root, max_ai_tracks_per_genre=MAX_AI_TRACKS_PER_GENRE, strict_single_label=STRICT_SINGLE_LABEL):
    records = []
    for batch_dir in sorted(p for p in ai_root.iterdir() if p.is_dir() and p.name.startswith("batch_")):
        try:
            ds = load_from_disk(str(batch_dir))
        except Exception:
            continue
        meta = ds.select_columns(["id", "tags", "duration", "model_name", "status"])
        for row_index in range(len(meta)):
            row = meta[row_index]
            mapped_genre = map_tags_to_gtzan_genre(row["tags"], strict_single_label=strict_single_label)
            if mapped_genre is None:
                continue
            records.append({
                "batch": batch_dir.name,
                "row_index": row_index,
                "track_id": str(row["id"]),
                "tags": row["tags"],
                "duration": row["duration"],
                "model_name": row["model_name"],
                "status": row["status"],
                "mapped_genre": mapped_genre,
            })
    metadata_df = pd.DataFrame(records).sort_values(["mapped_genre", "batch", "row_index"]).reset_index(drop=True)
    if max_ai_tracks_per_genre is not None and not metadata_df.empty:
        metadata_df = metadata_df.groupby("mapped_genre", group_keys=False).head(max_ai_tracks_per_genre).reset_index(drop=True)
    return metadata_df


def extract_mfcc_segments_from_audio_bytes(audio_bytes, suffix):
    with tempfile.NamedTemporaryFile(suffix=suffix or ".mp3", delete=True) as tmp_file:
        tmp_file.write(audio_bytes)
        tmp_file.flush()
        return extract_mfcc_segments_from_file(tmp_file.name)


def build_ai_eval_json(metadata_df, ai_root, output_json_path):
    data = {
        "mapping": label_names,
        "labels": [],
        "mfcc": [],
        "track_keys": [],
        "track_ids": [],
        "track_genres": [],
    }
    for batch_name, batch_rows in metadata_df.groupby("batch"):
        batch_ds = load_from_disk(str(ai_root / batch_name)).cast_column("audio", Audio(decode=False))
        for record in batch_rows.to_dict("records"):
            row = batch_ds[int(record["row_index"])]
            audio_info = row["audio"]
            audio_bytes = audio_info.get("bytes")
            if audio_bytes is None:
                continue
            suffix = Path(audio_info.get("path", "track.mp3")).suffix or ".mp3"
            segments = extract_mfcc_segments_from_audio_bytes(audio_bytes, suffix)
            if not segments:
                continue
            label_id = genre_to_id[record["mapped_genre"]]
            track_key = f"{record['batch']}:{record['row_index']}:{record['track_id']}"
            for mfcc in segments:
                data["mfcc"].append(mfcc.tolist())
                data["labels"].append(int(label_id))
                data["track_keys"].append(track_key)
                data["track_ids"].append(str(record["track_id"]))
                data["track_genres"].append(str(record["mapped_genre"]))
    with open(output_json_path, "w") as fp:
        json.dump(data, fp)
    return data


def load_ai_eval_json(json_path):
    with open(json_path, "r") as fp:
        data = json.load(fp)
    if "track_keys" not in data:
        data["track_keys"] = infer_track_keys_from_segment_order(data["track_ids"])
    X = np.array(data["mfcc"], dtype=np.float32)[..., np.newaxis]
    y = np.array(data["labels"], dtype=np.int64)
    return data, X, y


if RUN_AI_EVAL:
    ai_metadata_df = collect_ai_eval_metadata(AI_DATA_ROOT)
    ai_metadata_df.to_csv(AI_METADATA_CSV, index=False)
    print(f"Mapped AI tracks kept: {len(ai_metadata_df)}")
    if not ai_metadata_df.empty:
        display(ai_metadata_df["mapped_genre"].value_counts().sort_index().to_frame("tracks"))

        cached_ai_json_path = AI_JSON_PATH if AI_JSON_PATH.exists() else SOURCE_AI_JSON_PATH
        if cached_ai_json_path.exists() and not REBUILD_AI_FEATURES:
            print("Loading AI MFCC features:", cached_ai_json_path.relative_to(PROJECT_ROOT))
            ai_eval_data, X_ai, y_ai = load_ai_eval_json(cached_ai_json_path)
        else:
            ai_eval_data = build_ai_eval_json(ai_metadata_df, AI_DATA_ROOT, AI_JSON_PATH)
            ai_eval_data, X_ai, y_ai = load_ai_eval_json(AI_JSON_PATH)

        ai_pred = model.predict(X_ai, verbose=0).argmax(axis=1)
        print(f"AI segment-level accuracy: {accuracy_score(y_ai, ai_pred):.4f}")
        print(f"AI segment-level macro F1: {f1_score(y_ai, ai_pred, average='macro'):.4f}")
        print(classification_report(y_ai, ai_pred, labels=list(range(len(label_names))), target_names=label_names, zero_division=0))
        plot_confusion_matrix(y_ai, ai_pred, label_names, "AI Music Segment-Level Confusion Matrix")

        segment_results_df = pd.DataFrame({
            "track_key": ai_eval_data["track_keys"],
            "track_id": ai_eval_data["track_ids"],
            "true_label_id": y_ai,
            "pred_label_id": ai_pred,
            "true_label": [label_names[idx] for idx in y_ai],
            "pred_label": [label_names[idx] for idx in ai_pred],
        })
        segment_results_df.to_csv(AI_SEGMENT_RESULTS_CSV, index=False)

        track_results_df = (
            segment_results_df.groupby(["track_key", "track_id", "true_label"], as_index=False)
            .agg(
                true_label_id=("true_label_id", "first"),
                pred_label_id=("pred_label_id", lambda series: Counter(series).most_common(1)[0][0]),
                num_segments=("pred_label_id", "size"),
            )
        )
        track_results_df["pred_label"] = track_results_df["pred_label_id"].map(lambda idx: label_names[idx])
        track_results_df.to_csv(AI_TRACK_RESULTS_CSV, index=False)

        print(f"AI track-level accuracy: {accuracy_score(track_results_df['true_label_id'], track_results_df['pred_label_id']):.4f}")
        print(f"AI track-level macro F1: {f1_score(track_results_df['true_label_id'], track_results_df['pred_label_id'], average='macro'):.4f}")

        track_summary_df = (
            track_results_df.groupby(["true_label", "pred_label"], as_index=False)
            .size()
            .rename(columns={"size": "tracks"})
            .sort_values(["tracks", "true_label", "pred_label"], ascending=[False, True, True])
        )
        display(track_summary_df.head(10))
    else:
        print("No AI tracks matched the GTZAN genre mapping under the current settings.")
else:
    print("Skipping AI evaluation because the local Suno folder was not found.")

## Part 12: Novelty / Contributions: AI-Music Robust Inference

The parent-paper block makes one prediction per MFCC segment and then uses a hard majority vote for each AI track. Demo 6 keeps that Demo 5 comparison and adds a research-backed classifier block; the inference comparison replaces that single AI decision rule with a small comparison study:

- **Section-aware voting:** focus the track decision on the middle/core MFCC segments where the genre is often clearer than in AI-generated intros or transitions.
- **Probability averaging:** average the full softmax distribution across a track instead of only counting argmax labels.
- **Confidence-weighted averaging:** give more influence to segments where the model is less uncertain.
- **Overlapping-window test-time augmentation:** keep the same trained CNN, but evaluate more 6-second MFCC windows across each AI track so predictions are less sensitive to where the fixed segments begin.

This creates a reusable comparison table for the final paper's **Novelty** or **Contributions** section while preserving the parent-paper CNN as the baseline.


In [ ]:
CORE_WINDOW_START_INDEX = 2
CORE_WINDOW_STOP_INDEX = 4


def macro_f1_all_classes(y_true, y_pred):
    return f1_score(
        y_true,
        y_pred,
        average="macro",
        labels=list(range(len(label_names))),
        zero_division=0,
    )


def probability_frame_from_outputs(track_ids, y_true, probs, track_genres=None, track_keys=None):
    probs = np.asarray(probs, dtype=np.float32)
    y_true = np.asarray(y_true, dtype=np.int64)
    track_ids = [str(track_id) for track_id in track_ids]
    if track_keys is None:
        track_keys = track_ids
    track_keys = [str(track_key) for track_key in track_keys]

    pred_label_id = probs.argmax(axis=1).astype(np.int64)
    frame = pd.DataFrame(
        {
            "track_key": track_keys,
            "track_id": track_ids,
            "true_label_id": y_true,
            "pred_label_id": pred_label_id,
            "true_label": [label_names[idx] for idx in y_true],
            "pred_label": [label_names[idx] for idx in pred_label_id],
            "confidence": probs.max(axis=1),
        }
    )
    frame["view_position"] = frame.groupby("track_key").cumcount()

    clipped_probs = np.clip(probs, 1e-9, 1.0)
    normalized_entropy = -np.sum(clipped_probs * np.log(clipped_probs), axis=1) / np.log(len(label_names))
    frame["entropy_weight"] = 1.0 - normalized_entropy

    prob_cols = []
    for idx, label_name in enumerate(label_names):
        col = f"p_{idx}_{label_name}"
        frame[col] = probs[:, idx]
        prob_cols.append(col)
    return frame, prob_cols


def weighted_mean_probability(group, prob_cols, weight_col=None):
    group_probs = group[prob_cols].to_numpy(dtype=np.float64)
    if weight_col is None:
        return group_probs.mean(axis=0)

    weights = group[weight_col].to_numpy(dtype=np.float64)
    weights = np.where(np.isfinite(weights), weights, 0.0)
    weights = np.clip(weights, 1e-6, None)
    if weights.sum() <= 0:
        return group_probs.mean(axis=0)
    return np.average(group_probs, axis=0, weights=weights)


def aggregate_track_predictions(view_frame, prob_cols, strategy):
    records = []
    for track_key, group in view_frame.groupby("track_key", sort=False):
        if strategy == "majority_vote":
            pred_label_id = int(Counter(group["pred_label_id"]).most_common(1)[0][0])
            track_confidence = float(group["confidence"].mean())
        elif strategy == "outro_trimmed_majority":
            section_group = group[group["view_position"] < max(NUM_SEGMENTS - 1, 1)]
            if section_group.empty:
                section_group = group
            pred_label_id = int(Counter(section_group["pred_label_id"]).most_common(1)[0][0])
            track_confidence = float(section_group["confidence"].mean())
        elif strategy == "core_window_majority":
            section_group = group[
                (group["view_position"] >= CORE_WINDOW_START_INDEX)
                & (group["view_position"] < CORE_WINDOW_STOP_INDEX)
            ]
            if section_group.empty:
                section_group = group
            pred_label_id = int(Counter(section_group["pred_label_id"]).most_common(1)[0][0])
            track_confidence = float(section_group["confidence"].mean())
        elif strategy == "mean_probability":
            mean_probs = weighted_mean_probability(group, prob_cols)
            pred_label_id = int(np.argmax(mean_probs))
            track_confidence = float(np.max(mean_probs))
        elif strategy == "confidence_weighted_probability":
            mean_probs = weighted_mean_probability(group, prob_cols, weight_col="confidence")
            pred_label_id = int(np.argmax(mean_probs))
            track_confidence = float(np.max(mean_probs))
        elif strategy == "entropy_weighted_probability":
            mean_probs = weighted_mean_probability(group, prob_cols, weight_col="entropy_weight")
            pred_label_id = int(np.argmax(mean_probs))
            track_confidence = float(np.max(mean_probs))
        else:
            raise ValueError(f"Unknown aggregation strategy: {strategy}")

        true_label_id = int(group["true_label_id"].iloc[0])
        records.append(
            {
                "track_key": str(track_key),
                "track_id": str(group["track_id"].iloc[0]),
                "true_label_id": true_label_id,
                "pred_label_id": pred_label_id,
                "true_label": label_names[true_label_id],
                "pred_label": label_names[pred_label_id],
                "num_views": int(len(group)),
                "track_confidence": track_confidence,
            }
        )
    return pd.DataFrame(records)


def method_summary(method, method_group, evaluation_level, y_true, y_pred, item_count, mean_views, predict_seconds=0.0, feature_seconds=0.0):
    return {
        "method": method,
        "method_group": method_group,
        "evaluation_level": evaluation_level,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(macro_f1_all_classes(y_true, y_pred)),
        "items": int(item_count),
        "mean_views_per_track": float(mean_views),
        "feature_seconds": float(feature_seconds),
        "prediction_seconds": float(predict_seconds),
    }


def extract_overlapping_mfcc_views_from_audio_bytes(audio_bytes, suffix, num_views=TTA_NUM_VIEWS, window_seconds=TTA_WINDOW_SECONDS):
    with tempfile.NamedTemporaryFile(suffix=suffix or ".mp3", delete=True) as tmp_file:
        tmp_file.write(audio_bytes)
        tmp_file.flush()
        signal, sr = librosa.load(tmp_file.name, sr=SAMPLE_RATE, duration=TRACK_DURATION)

    window_samples = int(window_seconds * SAMPLE_RATE)
    expected_vectors = math.ceil(window_samples / HOP_LENGTH)
    if len(signal) < window_samples:
        signal = np.pad(signal, (0, window_samples - len(signal)))

    max_start = max(len(signal) - window_samples, 0)
    if num_views <= 1 or max_start == 0:
        starts = np.array([0], dtype=np.int64)
    else:
        starts = np.linspace(0, max_start, num=num_views, dtype=np.int64)

    views = []
    for view_index, start_sample in enumerate(np.unique(starts)):
        segment_signal = signal[start_sample : start_sample + window_samples]
        if len(segment_signal) < window_samples:
            segment_signal = np.pad(segment_signal, (0, window_samples - len(segment_signal)))
        mfcc = librosa.feature.mfcc(
            y=segment_signal,
            sr=sr,
            n_mfcc=NUM_MFCC,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
        ).T
        if len(mfcc) == expected_vectors:
            views.append((int(view_index), mfcc))
    return views


def build_ai_tta_json(metadata_df, ai_root, output_json_path):
    data = {
        "mapping": label_names,
        "labels": [],
        "mfcc": [],
        "track_keys": [],
        "track_ids": [],
        "track_genres": [],
        "view_index": [],
    }

    for batch_name, batch_rows in metadata_df.groupby("batch"):
        try:
            batch_ds = load_from_disk(str(ai_root / batch_name)).cast_column("audio", Audio(decode=False))
        except Exception as exc:
            print(f"SKIP Suno batch for TTA: {batch_name} | {type(exc).__name__}: {exc}")
            continue

        for record in batch_rows.to_dict("records"):
            row = batch_ds[int(record["row_index"])]
            audio_info = row["audio"]
            audio_bytes = audio_info.get("bytes")
            if audio_bytes is None:
                continue

            suffix = Path(audio_info.get("path", "track.mp3")).suffix or ".mp3"
            try:
                views = extract_overlapping_mfcc_views_from_audio_bytes(audio_bytes, suffix)
            except Exception as exc:
                print(f"SKIP AI TTA track: {record['track_id']} | {type(exc).__name__}: {exc}")
                continue

            label_id = genre_to_id[record["mapped_genre"]]
            track_key = f"{record['batch']}:{record['row_index']}:{record['track_id']}"
            for view_index, mfcc in views:
                data["mfcc"].append(mfcc.tolist())
                data["labels"].append(int(label_id))
                data["track_keys"].append(track_key)
                data["track_ids"].append(str(record["track_id"]))
                data["track_genres"].append(str(record["mapped_genre"]))
                data["view_index"].append(int(view_index))

    with open(output_json_path, "w") as fp:
        json.dump(data, fp)
    return data


def load_ai_tta_json(json_path):
    with open(json_path, "r") as fp:
        data = json.load(fp)
    X = np.array(data["mfcc"], dtype=np.float32)[..., np.newaxis]
    y = np.array(data["labels"], dtype=np.int64)
    return data, X, y


if RUN_AI_EVAL and "X_ai" in globals() and len(X_ai) > 0:
    comparison_rows = []
    track_prediction_frames = []

    prediction_start = time.perf_counter()
    baseline_probs = model.predict(X_ai, verbose=0)
    baseline_predict_seconds = time.perf_counter() - prediction_start

    baseline_track_ids = ai_eval_data["track_ids"]
    baseline_track_keys = ai_eval_data.get("track_keys", baseline_track_ids)
    baseline_view_frame, baseline_prob_cols = probability_frame_from_outputs(
        track_ids=baseline_track_ids,
        track_keys=baseline_track_keys,
        y_true=y_ai,
        probs=baseline_probs,
    )

    comparison_rows.append(
        method_summary(
            method="parent_segment_argmax",
            method_group="parent_baseline",
            evaluation_level="segment",
            y_true=y_ai,
            y_pred=baseline_view_frame["pred_label_id"].to_numpy(),
            item_count=len(baseline_view_frame),
            mean_views=baseline_view_frame.groupby("track_key").size().mean(),
            predict_seconds=baseline_predict_seconds,
        )
    )

    aggregation_strategies = {
        "parent_majority_vote": ("parent_baseline", "majority_vote"),
        "core_window_majority_vote": ("new_contribution", "core_window_majority"),
        "outro_trimmed_majority_vote": ("new_contribution", "outro_trimmed_majority"),
        "mean_probability_vote": ("new_contribution", "mean_probability"),
        "confidence_weighted_probability_vote": ("new_contribution", "confidence_weighted_probability"),
        "entropy_weighted_probability_vote": ("new_contribution", "entropy_weighted_probability"),
    }

    for method_name, (method_group, strategy) in aggregation_strategies.items():
        track_frame = aggregate_track_predictions(baseline_view_frame, baseline_prob_cols, strategy=strategy)
        track_frame["method"] = method_name
        track_prediction_frames.append(track_frame)
        comparison_rows.append(
            method_summary(
                method=method_name,
                method_group=method_group,
                evaluation_level="track",
                y_true=track_frame["true_label_id"].to_numpy(),
                y_pred=track_frame["pred_label_id"].to_numpy(),
                item_count=len(track_frame),
                mean_views=track_frame["num_views"].mean(),
                predict_seconds=baseline_predict_seconds,
            )
        )

    if RUN_AI_TTA_EVAL:
        tta_feature_seconds = 0.0
        if AI_TTA_JSON_PATH.exists() and not REBUILD_AI_TTA_FEATURES:
            print("Loading overlapping-window AI MFCC views:", AI_TTA_JSON_PATH.relative_to(PROJECT_ROOT))
        else:
            print("Building overlapping-window AI MFCC views for test-time augmentation...")
            tta_start = time.perf_counter()
            build_ai_tta_json(ai_metadata_df, AI_DATA_ROOT, AI_TTA_JSON_PATH)
            tta_feature_seconds = time.perf_counter() - tta_start

        tta_eval_data, X_tta, y_tta = load_ai_tta_json(AI_TTA_JSON_PATH)
        if len(X_tta) > 0:
            prediction_start = time.perf_counter()
            tta_probs = model.predict(X_tta, verbose=0)
            tta_predict_seconds = time.perf_counter() - prediction_start
            tta_view_frame, tta_prob_cols = probability_frame_from_outputs(
                track_ids=tta_eval_data["track_ids"],
                track_keys=tta_eval_data["track_keys"],
                y_true=y_tta,
                probs=tta_probs,
            )
            tta_view_frame["view_index"] = tta_eval_data["view_index"]
            tta_view_frame.to_csv(AI_TTA_VIEW_RESULTS_CSV, index=False)

            comparison_rows.append(
                method_summary(
                    method="overlap_tta_segment_argmax",
                    method_group="new_contribution",
                    evaluation_level="segment",
                    y_true=y_tta,
                    y_pred=tta_view_frame["pred_label_id"].to_numpy(),
                    item_count=len(tta_view_frame),
                    mean_views=tta_view_frame.groupby("track_key").size().mean(),
                    feature_seconds=tta_feature_seconds,
                    predict_seconds=tta_predict_seconds,
                )
            )

            tta_track_strategies = {
                "overlap_tta_mean_probability_vote": "mean_probability",
                "overlap_tta_confidence_weighted_vote": "confidence_weighted_probability",
            }
            for method_name, strategy in tta_track_strategies.items():
                track_frame = aggregate_track_predictions(tta_view_frame, tta_prob_cols, strategy=strategy)
                track_frame["method"] = method_name
                track_prediction_frames.append(track_frame)
                comparison_rows.append(
                    method_summary(
                        method=method_name,
                        method_group="new_contribution",
                        evaluation_level="track",
                        y_true=track_frame["true_label_id"].to_numpy(),
                        y_pred=track_frame["pred_label_id"].to_numpy(),
                        item_count=len(track_frame),
                        mean_views=track_frame["num_views"].mean(),
                        feature_seconds=tta_feature_seconds,
                        predict_seconds=tta_predict_seconds,
                    )
                )
        else:
            print("Overlapping-window TTA produced zero usable views.")

    comparison_df = pd.DataFrame(comparison_rows).sort_values(
        ["evaluation_level", "method_group", "accuracy", "macro_f1"],
        ascending=[True, True, False, False],
    ).reset_index(drop=True)
    comparison_df.to_csv(AI_METHOD_COMPARISON_CSV, index=False)

    if track_prediction_frames:
        all_track_predictions_df = pd.concat(track_prediction_frames, ignore_index=True)
        all_track_predictions_df.to_csv(AI_TRACK_METHOD_RESULTS_CSV, index=False)

    print("Saved AI method comparison:", AI_METHOD_COMPARISON_CSV.relative_to(PROJECT_ROOT))
    display(comparison_df)

    track_comparison_df = comparison_df[comparison_df["evaluation_level"] == "track"].copy()
    if not track_comparison_df.empty:
        best_row = track_comparison_df.sort_values(["accuracy", "macro_f1"], ascending=False).iloc[0]
        print(
            "Best AI track method: "
            f"{best_row['method']} | accuracy={best_row['accuracy']:.4f} | "
            f"macro_f1={best_row['macro_f1']:.4f}"
        )
else:
    print("Skipping novelty comparison because AI evaluation did not produce usable MFCC segments.")


## Part 13: Research-Backed Method: MFCC Statistics Plus Linear SVM

I looked up several research directions before choosing what to implement here:

- **Feature-engineered SVM models:** music-genre work continues to use MFCC/timbral statistics with SVM-style classifiers, and recent GTZAN work also reports strong gains when tonal features and active learning are added to SVM/KNN/XGBoost baselines ([Velazquez-Lopez et al., 2024](https://rcs.cic.ipn.mx/2024_153_11/Enhancing%20Music%20Genre%20Classification%20Using%20Tonnetz%20and%20Active%20Learning.pdf); [Kour and Mehan, 2015](https://www.ijcaonline.org/archives/volume112/number6/19669-1119/)).
- **Feature fusion:** MFCC-derived high-level and low-level timbral descriptors have been combined with SVM/DNN classifiers, with the paper reporting an additional improvement from descriptor fusion ([Rajan et al., 2021](https://infocomp.dcc.ufla.br/index.php/infocomp/article/view/1604)).
- **Deep augmentation and mel-spectrogram CNNs:** recent GTZAN work reports strong results from mel spectrograms and augmentation such as splitting, pitch shifting, and noise ([Ba et al., 2025](https://www.sciencedirect.com/science/article/abs/pii/S1875952125000096)).
- **SpecAugment and pretrained audio models:** SpecAugment masks time/frequency regions of feature inputs ([Park et al., 2019](https://arxiv.org/abs/1904.08779)), while PANNs transfer large AudioSet-pretrained audio models to downstream audio tasks ([Kong et al., 2020](https://arxiv.org/abs/1912.10211)).

For Demo 6, I implemented the SVM/statistical-feature direction because it fits the existing dependencies, runs quickly on the current cached MFCC JSON, and gives a clean comparison against the parent CNN. The model compresses each MFCC segment into robust summary statistics, adds delta-MFCC statistics, then compares linear SVM, RBF SVM, logistic regression, and Extra Trees.


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC


def build_mfcc_statistics_features(mfcc_tensor):
    mfcc_array = np.asarray(mfcc_tensor, dtype=np.float32)
    if mfcc_array.ndim == 4:
        mfcc_array = mfcc_array[..., 0]

    deltas = np.diff(mfcc_array, axis=1)
    feature_blocks = [
        mfcc_array.mean(axis=1),
        mfcc_array.std(axis=1),
        np.percentile(mfcc_array, 25, axis=1),
        np.percentile(mfcc_array, 75, axis=1),
        mfcc_array.min(axis=1),
        mfcc_array.max(axis=1),
        deltas.mean(axis=1),
        deltas.std(axis=1),
    ]
    return np.nan_to_num(np.concatenate(feature_blocks, axis=1), copy=False)


def aggregate_label_predictions(track_keys, track_ids, y_true, y_pred, strategy="majority_vote"):
    prediction_frame = pd.DataFrame(
        {
            "track_key": [str(track_key) for track_key in track_keys],
            "track_id": [str(track_id) for track_id in track_ids],
            "true_label_id": np.asarray(y_true, dtype=np.int64),
            "pred_label_id": np.asarray(y_pred, dtype=np.int64),
        }
    )
    prediction_frame["view_position"] = prediction_frame.groupby("track_key").cumcount()

    records = []
    for track_key, group in prediction_frame.groupby("track_key", sort=False):
        if strategy == "outro_trimmed_majority":
            selected = group[group["view_position"] < max(NUM_SEGMENTS - 1, 1)]
            if selected.empty:
                selected = group
        else:
            selected = group

        pred_label_id = int(Counter(selected["pred_label_id"]).most_common(1)[0][0])
        true_label_id = int(group["true_label_id"].iloc[0])
        records.append(
            {
                "track_key": str(track_key),
                "track_id": str(group["track_id"].iloc[0]),
                "true_label_id": true_label_id,
                "pred_label_id": pred_label_id,
                "true_label": label_names[true_label_id],
                "pred_label": label_names[pred_label_id],
                "num_views": int(len(group)),
                "aggregation": strategy,
            }
        )
    return pd.DataFrame(records)


def all_class_macro_f1(y_true, y_pred):
    return f1_score(
        y_true,
        y_pred,
        labels=list(range(len(label_names))),
        average="macro",
        zero_division=0,
    )


if RUN_AI_EVAL and "X_ai" in globals() and len(X_ai) > 0:
    X_train_stats = build_mfcc_statistics_features(X_train)
    X_test_stats = build_mfcc_statistics_features(X_test)
    X_ai_stats = build_mfcc_statistics_features(X_ai)

    ai_track_keys = ai_eval_data.get("track_keys", infer_track_keys_from_segment_order(ai_eval_data["track_ids"]))
    ai_track_ids = ai_eval_data["track_ids"]

    candidate_models = {
        "mfcc_stats_linear_svm": make_pipeline(
            StandardScaler(),
            LinearSVC(C=0.5, max_iter=10000, dual="auto", random_state=SEED),
        ),
        "mfcc_stats_rbf_svm": make_pipeline(
            StandardScaler(),
            SVC(C=10.0, gamma="scale", random_state=SEED),
        ),
        "mfcc_stats_logistic_regression": make_pipeline(
            StandardScaler(),
            LogisticRegression(C=1.0, max_iter=3000, random_state=SEED),
        ),
        "mfcc_stats_extra_trees": ExtraTreesClassifier(
            n_estimators=400,
            max_features="sqrt",
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1,
        ),
    }

    research_rows = []
    segment_prediction_frames = []
    track_prediction_frames = []

    for model_name, estimator in candidate_models.items():
        fit_start = time.perf_counter()
        estimator.fit(X_train_stats, y_train)
        fit_seconds = time.perf_counter() - fit_start

        predict_start = time.perf_counter()
        gtzan_test_pred = estimator.predict(X_test_stats)
        ai_segment_pred = estimator.predict(X_ai_stats)
        predict_seconds = time.perf_counter() - predict_start

        segment_prediction_frames.append(
            pd.DataFrame(
                {
                    "model": model_name,
                    "track_key": ai_track_keys,
                    "track_id": ai_track_ids,
                    "true_label_id": y_ai,
                    "pred_label_id": ai_segment_pred,
                    "true_label": [label_names[idx] for idx in y_ai],
                    "pred_label": [label_names[idx] for idx in ai_segment_pred],
                }
            )
        )

        research_rows.append(
            {
                "method": model_name,
                "method_group": "research_mfcc_stats_ml",
                "evaluation_level": "gtzan_segment_test",
                "aggregation": "segment_argmax",
                "accuracy": float(accuracy_score(y_test, gtzan_test_pred)),
                "macro_f1": float(all_class_macro_f1(y_test, gtzan_test_pred)),
                "items": int(len(y_test)),
                "mean_views_per_track": 1.0,
                "fit_seconds": float(fit_seconds),
                "prediction_seconds": float(predict_seconds),
            }
        )
        research_rows.append(
            {
                "method": model_name,
                "method_group": "research_mfcc_stats_ml",
                "evaluation_level": "ai_segment",
                "aggregation": "segment_argmax",
                "accuracy": float(accuracy_score(y_ai, ai_segment_pred)),
                "macro_f1": float(all_class_macro_f1(y_ai, ai_segment_pred)),
                "items": int(len(y_ai)),
                "mean_views_per_track": 1.0,
                "fit_seconds": float(fit_seconds),
                "prediction_seconds": float(predict_seconds),
            }
        )

        for aggregation in ["majority_vote", "outro_trimmed_majority"]:
            track_frame = aggregate_label_predictions(
                track_keys=ai_track_keys,
                track_ids=ai_track_ids,
                y_true=y_ai,
                y_pred=ai_segment_pred,
                strategy=aggregation,
            )
            track_frame["method"] = model_name
            track_prediction_frames.append(track_frame)
            research_rows.append(
                {
                    "method": f"{model_name}_{aggregation}",
                    "method_group": "research_mfcc_stats_ml",
                    "evaluation_level": "ai_track",
                    "aggregation": aggregation,
                    "accuracy": float(accuracy_score(track_frame["true_label_id"], track_frame["pred_label_id"])),
                    "macro_f1": float(all_class_macro_f1(track_frame["true_label_id"], track_frame["pred_label_id"])),
                    "items": int(len(track_frame)),
                    "mean_views_per_track": float(track_frame["num_views"].mean()),
                    "fit_seconds": float(fit_seconds),
                    "prediction_seconds": float(predict_seconds),
                }
            )

    research_comparison_df = pd.DataFrame(research_rows)

    previous_comparison_df = pd.DataFrame()
    if "comparison_df" in globals() and not comparison_df.empty:
        previous_comparison_df = comparison_df.copy()
        if "aggregation" not in previous_comparison_df:
            previous_comparison_df["aggregation"] = "see_method"
        previous_comparison_df = previous_comparison_df.rename(columns={"feature_seconds": "feature_or_fit_seconds"})
        research_comparison_for_merge = research_comparison_df.rename(columns={"fit_seconds": "feature_or_fit_seconds"})
        all_method_comparison_df = pd.concat(
            [previous_comparison_df, research_comparison_for_merge],
            ignore_index=True,
            sort=False,
        )
    else:
        all_method_comparison_df = research_comparison_df.rename(columns={"fit_seconds": "feature_or_fit_seconds"})

    all_method_comparison_df = all_method_comparison_df.sort_values(
        ["evaluation_level", "accuracy", "macro_f1"],
        ascending=[True, False, False],
    ).reset_index(drop=True)
    all_method_comparison_df.to_csv(RESEARCH_METHOD_COMPARISON_CSV, index=False)

    if segment_prediction_frames:
        pd.concat(segment_prediction_frames, ignore_index=True).to_csv(RESEARCH_SEGMENT_RESULTS_CSV, index=False)
    if track_prediction_frames:
        pd.concat(track_prediction_frames, ignore_index=True).to_csv(RESEARCH_TRACK_RESULTS_CSV, index=False)

    print("Saved research method comparison:", RESEARCH_METHOD_COMPARISON_CSV.relative_to(PROJECT_ROOT))
    display(all_method_comparison_df)

    ai_track_research_df = research_comparison_df[research_comparison_df["evaluation_level"] == "ai_track"].copy()
    best_research_row = ai_track_research_df.sort_values(["accuracy", "macro_f1"], ascending=False).iloc[0]
    print(
        "Best research-backed AI track method: "
        f"{best_research_row['method']} | accuracy={best_research_row['accuracy']:.4f} | "
        f"macro_f1={best_research_row['macro_f1']:.4f}"
    )
else:
    print("Skipping research-backed comparison because AI evaluation did not produce usable MFCC segments.")


## Closing Note

`demo_6` now tells the full story end to end. The early sections reproduce the paper's longer exploratory preprocessing chain, and the later sections implement the same MFCC-plus-CNN modeling pipeline used in the replication.

That means this notebook is useful both for explaining the signal-processing logic in your report and for showing runnable model implementations, a concrete novelty table, and a research-backed classifier comparison for the final paper.